<a href="https://colab.research.google.com/github/samarreguigui/Computerlinguistik/blob/main/exercice_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from torch.nn.utils.rnn import pad_sequence
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import argparse


In [4]:
def read_data(path):
    pairs = []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            score_text = line.split("\t", 1)
            score = float(score_text[0])
            tokens = score_text[1].split()
            pairs.append((score, tokens))
    return pairs


In [7]:
class Vocabulary:
    def __init__(self, data, min_freq):
        counts = {}
        for rating, tokens in data:
            for tok in tokens:
                counts[tok] = counts.get(tok, 0) + 1

        self.stoi = {}
        self.stoi["<unk>"] = 0
        self.stoi["<pad>"] = 1

        idx = 2
        for tok, c in counts.items():
            if c >= min_freq:
                self.stoi[tok] = idx
                idx += 1

    def __call__(self, tokens):
        ids = []
        for tok in tokens:
            ids.append(self.stoi.get(tok, 0))
        return ids

    def __len__(self):
        return len(self.stoi)


In [8]:

def collate(batch, vocab):
    Bewertungen = []
    Wort_IDS = []
    Textlaengen = []

    for rating, tokens in batch:
        Bewertungen.append(rating)

        ids = vocab(tokens)
        Wort_IDS.append(torch.tensor(ids, dtype=torch.long))

        Textlaengen.append(len(ids))

    Bewertungen = torch.tensor(Bewertungen, dtype=torch.float32)
    Textlaengen = torch.tensor(Textlaengen, dtype=torch.long)

    Wort_IDS = pad_sequence(Wort_IDS, batch_first=True, padding_value=1)

    return Bewertungen, Wort_IDS, Textlaengen


In [12]:
#DataLoader Objekte erzeugen

train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    collate_fn=lambda batch: collate(batch, vocab)
)

dev_loader = DataLoader(
    dev_data,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda batch: collate(batch, vocab)
)


In [15]:
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for Bewertungen, Wort_IDS, Textlaengen in dataloader:
        Bewertungen = Bewertungen.to(device)
        Wort_IDS = Wort_IDS.to(device)
        Textlaengen = Textlaengen.to(device)

        optimizer.zero_grad()

        predictions = model(Wort_IDS, Textlaengen).squeeze(1)

        loss = criterion(predictions, Bewertungen)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)



In [16]:
import math
import torch

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for Bewertungen, Wort_IDS, Textlaengen in dataloader:
            Bewertungen = Bewertungen.to(device)
            Wort_IDS = Wort_IDS.to(device)
            Textlaengen = Textlaengen.to(device)

            predictions = model(Wort_IDS, Textlaengen).squeeze(1)

            loss = criterion(predictions, Bewertungen)
            total_loss += loss.item()

    mse = total_loss / len(dataloader)
    rmse = math.sqrt(mse)
    return rmse


In [18]:
import torch
import torch.nn as nn

class SentimentPrediction(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim):
        super().__init__()

        self.emb = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=emb_dim,
            padding_idx=1
        )

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.out = nn.Linear(hidden_dim * 2, 1)

    def forward(self, Wort_IDS, Textlaengen):
        embedded = self.emb(Wort_IDS)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            Textlaengen.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        pooled = torch.mean(output, dim=1)

        raw = self.out(pooled)

        prediction = torch.sigmoid(raw) * 4 + 1

        return prediction


In [29]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_file = "/content/sentiment.train.tsv"
    dev_file = "/content/sentiment.dev.tsv"
    parfile = "/content/bestmodel.pt"

    train_data = read_data(train_file)
    dev_data = read_data(dev_file)

    vocab = Vocabulary(train_data, min_freq=2)

    train_loader = DataLoader(
        train_data,
        batch_size=32,
        shuffle=True,
        collate_fn=lambda batch: collate(batch, vocab)
    )

    dev_loader = DataLoader(
        dev_data,
        batch_size=32,
        shuffle=False,
        collate_fn=lambda batch: collate(batch, vocab)
    )

    model = SentimentPrediction(
        vocab_size=len(vocab),
        emb_dim=200,
        hidden_dim=64
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    best_rmse = float("inf")
    best_state = None
    epochs = 10

    for epoch in range(epochs):
        train_loss = train(model, train_loader, optimizer, criterion, device)
        rmse = evaluate(model, dev_loader, criterion, device)

        print("Epoche:", epoch + 1, "Train Loss:", train_loss, "RMSE:", rmse)

        if rmse < best_rmse:
            best_rmse = rmse
            best_state = model.state_dict()

    torch.save(best_state, parfile)

main()


Epoche: 1 Train Loss: 1.5104468040698475 RMSE: 1.1430664144441383
Epoche: 2 Train Loss: 1.0877719138668718 RMSE: 1.0822869284522738
Epoche: 3 Train Loss: 0.7413224880391739 RMSE: 1.0832948456903604
Epoche: 4 Train Loss: 0.5254656499467986 RMSE: 1.0914307313516438
Epoche: 5 Train Loss: 0.38216287757127027 RMSE: 1.111030863340939
Epoche: 6 Train Loss: 0.2933877071693595 RMSE: 1.1239021318195097
Epoche: 7 Train Loss: 0.23574399063538076 RMSE: 1.1124719577947315
Epoche: 8 Train Loss: 0.19504718711313684 RMSE: 1.1319981004986912
Epoche: 9 Train Loss: 0.17106801946734668 RMSE: 1.1072608459685123
Epoche: 10 Train Loss: 0.15013607017947048 RMSE: 1.1196621577671608
